# ViYT-Diar: automatic baseline / oracle_num_speakers

**Нет обучения, подбора параметров и удаления моделей.**

1. Импортируйте notebook в новую Kaggle-сессию. Подключите обновлённый code dataset и существующий `model_0034.model`.
   Можно подключить небольшой `viyt-oracle-code.zip` как новый dataset рядом со старым `ecapa-tdnn`: веса повторно загружать не нужно.
2. Для oracle подключите также **исходный ZIP результатов automatic** как Kaggle dataset
   (после распаковки внутри должен быть `results/manifest.json`). В конфиге поставьте `SPEAKER_MODE = 'oracle'`.
   Automatic остаётся режимом по умолчанию.
3. Internet ON, GPU T4; **Run All**. В конце скачайте ZIP. Каждый запуск сохраняется отдельно.

Оцениваются все **100 test записей** [ViYT-Diar](https://huggingface.co/datasets/tuanduy1612/ViYT-Diar),
revision `c66b118709d68c7538408d086232d850356db374`; около 308 MB скачивания.
Oracle сообщает AHC только `len(set(sample['speaker']))`. Имена и интервалы reference в inference не передаются.
AHC получает fixed `n_clusters`; threshold=0.65 остаётся в конфиге baseline, но **не используется** для выбора числа кластеров в oracle.

Остальные условия прежние: модель 34, VAD=0.5, window=1.5 s, step=0.75 s, min_segment=0.5 s,
seed=42, collar=0.25 s, overlap учитывается, preprocessing/postprocessing/DER без изменений.
Перед oracle сверяются baseline SHA модели, записи/WAV/reference, параметры и версии численных библиотек.
При несовпадении прогон остановится с пояснением; не обходите проверку для заявления о чистом сравнении.


In [ ]:
from pathlib import Path

# Обычно менять пути не требуется. Укажите их явно при нескольких копиях dataset.
SOURCE_OVERRIDE = None  # например Path('/kaggle/input/ecapa-tdnn/ecapa-tdnn-kaggle')
MODEL_OVERRIDE = None   # полный путь к model_0034.model, если он вне папки проекта
DATASET_ID = 'tuanduy1612/ViYT-Diar'
DATASET_REVISION = 'c66b118709d68c7538408d086232d850356db374'
SPEAKER_MODE = 'automatic'  # 'automatic' или 'oracle'
# Для oracle подключите ZIP результатов baseline как отдельный Kaggle dataset.
BASELINE_RESULTS = None  # auto-search; либо Path('/kaggle/input/.../results')
if SPEAKER_MODE not in {'automatic', 'oracle'}:
    raise ValueError('SPEAKER_MODE must be automatic or oracle')
MODE_NAME = 'oracle_reference_count' if SPEAKER_MODE == 'oracle' else 'automatic'
SEED = 42
DEVICE = 'auto'
COLLAR = 0.25
SKIP_OVERLAP = False
PIPELINE = dict(method='ahc', threshold=0.65, vad_threshold=0.5,
                window_sec=1.5, step_sec=0.75, max_speakers=10, min_segment_sec=0.5)


In [ ]:
import os
import sys
import shutil
import tempfile
import subprocess
import importlib.metadata
import json

input_root = Path('/kaggle/input')
required = ['models/ecapa_tdnn.py', 'diarize/pipeline.py', 'eval/der.py',
            'utils/audio.py', 'utils/model.py', 'utils/provenance.py',
            'experiments/viyt_diagnostics.py', 'requirements.txt']
if SOURCE_OVERRIDE is None:
    candidates = sorted({p.parent.parent for p in input_root.rglob('eval/der.py')
                         if all((p.parent.parent / name).is_file() for name in required)})
    if len(candidates) != 1:
        raise RuntimeError(f'Нужна одна обновлённая копия проекта. Найдены: {candidates}. '
                           'Подключите ecapa-tdnn или задайте SOURCE_OVERRIDE.')
    SOURCE = candidates[0]
else:
    SOURCE = Path(SOURCE_OVERRIDE)
if not all((SOURCE / name).is_file() for name in required):
    raise FileNotFoundError(f'Неполный/устаревший проект: {SOURCE}. Обновите code dataset.')

if MODEL_OVERRIDE is not None:
    MODEL = Path(MODEL_OVERRIDE)
else:
    models = sorted(SOURCE.rglob('model_0034.model'))
    if not models:
        models = sorted(input_root.rglob('model_0034.model'))
    if len(models) != 1:
        raise RuntimeError(f'Нужен model_0034.model. Найдены: {models}. Задайте MODEL_OVERRIDE.')
    MODEL = models[0]
if not MODEL.is_file() or MODEL.name != 'model_0034.model':
    raise FileNotFoundError(f'Требуется существующий model_0034.model: {MODEL}')

# Новый каталог при каждом запуске: старые результаты и модели сохраняются.
RUN = Path(tempfile.mkdtemp(prefix=f'viyt_model0034_{SPEAKER_MODE}_', dir='/kaggle/working'))
CODE = RUN / 'code'
CODE.mkdir()
for folder in ('models', 'diarize', 'eval', 'utils', 'experiments'):
    shutil.copytree(SOURCE / folder, CODE / folder,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
shutil.copy2(SOURCE / 'requirements.txt', CODE / 'requirements.txt')
os.chdir(CODE)
sys.path.insert(0, str(CODE))

# Baseline is read-only. Kaggle normally expands the uploaded results ZIP.
if SPEAKER_MODE == 'oracle':
    if BASELINE_RESULTS is None:
        baseline_candidates = []
        for path in input_root.rglob('manifest.json'):
            if not (path.parent / 'der_results.json').is_file():
                continue
            saved = json.loads(path.read_text(encoding='utf-8'))
            if saved.get('speaker_mode') == 'automatic' and saved.get('status') == 'complete':
                baseline_candidates.append(path.parent)
        if len(baseline_candidates) != 1:
            raise RuntimeError(f'Подключите baseline results dataset или задайте BASELINE_RESULTS: {baseline_candidates}')
        BASELINE_RESULTS = baseline_candidates[0]
    BASELINE_RESULTS = Path(BASELINE_RESULTS)
    baseline_config = json.loads((BASELINE_RESULTS / 'manifest.json').read_text(encoding='utf-8'))

# Сохраняем предустановленные CUDA-сборки Kaggle, устанавливаем остальные зависимости.
constraints = RUN / 'torch-constraints.txt'
constraints.write_text(''.join(f'{name}=={importlib.metadata.version(name)}\n'
                               for name in ('torch', 'torchaudio')), encoding='utf-8')
if SPEAKER_MODE == 'oracle':
    # Pin the reported numerical/scoring dependencies to the baseline versions.
    with constraints.open('a', encoding='utf-8') as handle:
        for name, version in baseline_config['environment']['packages'].items():
            if name not in {'torch', 'torchaudio'} and version is not None:
                handle.write(f'{name}=={version}\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt',
                '-c', str(constraints)], check=True)
print('Code:', SOURCE, '\nModel:', MODEL, '\nRun:', RUN)


In [ ]:
import json
import torch
from datasets import load_dataset, Audio
from huggingface_hub import HfApi
from utils.seed import set_seed
from utils.device import resolve_device
from utils.provenance import sha256, environment

set_seed(SEED)
device = str(resolve_device(DEVICE))
revision = HfApi().dataset_info(DATASET_ID, revision=DATASET_REVISION).sha
# decode=False: байты декодируются soundfile, без зависимости от TorchCodec.
dataset = load_dataset(DATASET_ID, split='test', revision=revision)
dataset = dataset.cast_column('audio', Audio(decode=False))
expected_columns = {'audio', 'audio_id', 'speaker', 'timestamps_start', 'timestamps_end'}
if not expected_columns.issubset(dataset.column_names) or len(dataset) == 0:
    raise ValueError(f'Неожиданная схема ViYT-Diar: {dataset.features}')

if len(dataset) != 100:
    raise ValueError(f'Expected 100 test recordings, got {len(dataset)}')

RESULTS = RUN / 'results' / ('oracle_num_speakers' if SPEAKER_MODE == 'oracle' else 'baseline')
RESULTS.mkdir(parents=True)
manifest = dict(dataset=DATASET_ID, revision=revision, split='test', files_expected=len(dataset),
                model=str(MODEL), model_sha256=sha256(MODEL), seed=SEED, device=device,
                speaker_mode=MODE_NAME, experiment='oracle_num_speakers' if SPEAKER_MODE == 'oracle' else 'baseline',
                ahc_threshold_active=SPEAKER_MODE == 'automatic', pipeline=PIPELINE, collar=COLLAR,
                skip_overlap=SKIP_OVERLAP, environment=environment(),
                cuda_version=torch.version.cuda,
                gpu=torch.cuda.get_device_name() if device.startswith('cuda') else None,
                code_sha256={p.relative_to(CODE).as_posix(): sha256(p) for p in sorted(CODE.rglob('*.py'))},
                status='started', rate_units='percent', duration_units='seconds')
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True)
(RESULTS / 'pip-freeze.txt').write_text(freeze.stdout, encoding='utf-8')
print('Dataset:', DATASET_ID, '\nRevision:', revision, '\nFiles:', len(dataset), '\nDevice:', device)


In [ ]:
import math
import re
import soundfile as sf
from utils.audio import decode_audio

def export_record(row, audio_dir, ref_dir):
    """Convert the documented ViYT-Diar row; preserve all annotated overlap."""
    file_id = row['audio_id']
    if not isinstance(file_id, str) or not re.fullmatch(r'[A-Za-z0-9_-]+', file_id):
        raise ValueError(f'Unsafe/unsupported audio_id: {file_id!r}')
    wav_path, rttm_path = audio_dir / f'{file_id}.wav', ref_dir / f'{file_id}.rttm'
    if wav_path.exists() or rttm_path.exists():
        raise ValueError(f'Duplicate audio_id or existing export: {file_id}')
    speakers, starts, ends = row['speaker'], row['timestamps_start'], row['timestamps_end']
    if not (len(speakers) == len(starts) == len(ends)):
        raise ValueError(f'{file_id}: annotation lists have different lengths')
    audio = decode_audio(row['audio'])
    duration = len(audio) / 16000
    segments = []
    for speaker, start, end in zip(speakers, starts, ends):
        start, end = float(start), float(end)
        if (not isinstance(speaker, str) or not speaker or any(c.isspace() for c in speaker)
                or not math.isfinite(start) or not math.isfinite(end)
                or not 0 <= start < end or end > duration + 0.002):
            raise ValueError(f'{file_id}: invalid annotation {(speaker, start, end)}, duration={duration}')
        segments.append((start, end, speaker))
    lines = [f'SPEAKER {file_id} 1 {start:.6f} {end-start:.6f} <NA> <NA> {speaker} <NA> <NA>'
             for start, end, speaker in sorted(segments)]
    # FLOAT avoids an extra PCM quantization when converting decoded audio.
    sf.write(wav_path, audio, 16000, subtype='FLOAT')
    rttm_path.write_text(''.join(line + '\n' for line in lines), encoding='utf-8')
    return dict(file_id=file_id, duration=duration, segments=len(segments),
                speakers_ref=len(set(speakers)), audio_sha256=sha256(wav_path),
                reference_sha256=sha256(rttm_path))


In [ ]:
from tqdm.auto import tqdm

AUDIO = RUN / 'audio'
REFERENCE = RESULTS / 'reference_rttm'
HYPOTHESIS = RESULTS / 'hypothesis_rttm'
for directory in (AUDIO, REFERENCE, HYPOTHESIS):
    directory.mkdir()
records = [export_record(row, AUDIO, REFERENCE) for row in tqdm(dataset, desc='Export WAV + RTTM')]
(RESULTS / 'recordings.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
print('Exported:', len(records), 'files;', sum(row['duration'] for row in records) / 60, 'minutes')


## Диаризация и оценка

Модель загружается один раз. Automatic передаёт `n_speakers=None`, oracle — только целое число из reference.
Недостаточно embedding windows для заданного числа кластеров — понятная ошибка с audio_id,
manifest `failed`, без публикации неполного corpus результата. Automatic с пустым VAD сохраняет прежний пустой hypothesis.

Число кластеров проверяется **до** сглаживания. Постобработка может убрать короткий кластер;
мы её не меняем ради искусственных 100% accuracy. В `diagnostics.csv` есть counts до/после,
число окон и размеры кластеров. Итоговый N_pred считается по сохранённому RTTM.

`pyannote.metrics` оценивает полную длительность WAV. Corpus DER вычисляется из сумм секунд ошибок
и scored reference speaker-time; matching labels остаётся permutation-invariant.
Pipeline не предсказывает overlapping speech, но reference overlap сохраняется и учитывается.
В диагностике `reference_speech_duration` и `predicted_speech_duration` — объединение речевых интервалов
**до collar**, без двойного счёта overlap. `scored_reference_speaker_time` — знаменатель DER после scoring exclusions.
DER и компоненты — проценты; speaker-count accuracy — доля 0..1.


In [ ]:
from experiments.viyt_diagnostics import validate_baseline

baseline_metrics = None
if SPEAKER_MODE == 'oracle':
    if any(record['speakers_ref'] < 1 for record in records):
        raise ValueError('Oracle requires at least one reference speaker per recording')
    baseline_manifest, baseline_metrics = validate_baseline(BASELINE_RESULTS, manifest, records)
    manifest['baseline_manifest_sha256'] = sha256(BASELINE_RESULTS / 'manifest.json')
    manifest['baseline_pipeline_sha256'] = baseline_manifest['code_sha256']['diarize/pipeline.py']
    (RESULTS / 'baseline_manifest.json').write_text(json.dumps(baseline_manifest, indent=2), encoding='utf-8')
    (RESULTS / 'baseline_der_results.json').write_text(json.dumps(baseline_metrics, indent=2), encoding='utf-8')
    print('Baseline validated: identical model, data, reference, parameters and reported numerical environment.')
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')


In [ ]:
from utils.model import load_model
from diarize.pipeline import diarize

encoder = load_model(str(MODEL), device)
traces = {}
for record in tqdm(records, desc=f'Diarization model 34: {MODE_NAME}'):
    file_id = record['file_id']
    # This integer is the ONLY reference information passed into inference.
    count = record['speakers_ref'] if SPEAKER_MODE == 'oracle' else None
    traces[file_id] = {}
    try:
        diarize(str(AUDIO / f'{file_id}.wav'), str(MODEL),
                output_rttm=str(HYPOTHESIS / f'{file_id}.rttm'),
                encoder=encoder, device=device, seed=SEED, n_speakers=count,
                diagnostics=traces[file_id], **PIPELINE)
    except (ValueError, RuntimeError) as error:
        manifest.update(status='failed', failed_audio_id=file_id, error=str(error))
        (RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
        (RESULTS / 'clustering_diagnostics.json').write_text(json.dumps(traces, indent=2), encoding='utf-8')
        raise
(RESULTS / 'clustering_diagnostics.json').write_text(json.dumps(traces, indent=2), encoding='utf-8')
print('Inference finished:', len(records), 'recordings')


In [ ]:
import csv
from eval.der import load_rttm, evaluate_annotations, save_results, print_results

expected_ids = {record['file_id'] for record in records}
actual_ids = {path.stem for path in HYPOTHESIS.glob('*.rttm')}
if actual_ids != expected_ids or len(records) != len(dataset):
    raise RuntimeError('Неполный набор файлов: corpus result не сохранён.')
references = load_rttm(str(REFERENCE))
hypotheses = load_rttm(str(HYPOTHESIS), allow_empty=True)
results = evaluate_annotations(references, hypotheses, collar=COLLAR,
                               skip_overlap=SKIP_OVERLAP,
                               durations={r['file_id']: r['duration'] for r in records})
save_results(results, output=str(RESULTS / 'der_results.json'), csv_path=str(RESULTS / 'der_results.csv'))
print_results(results)
total = results['__total__']
summary = dict(model=MODEL.name, dataset=DATASET_ID, split='test', method=PIPELINE['method'],
               speaker_mode=MODE_NAME, ahc_threshold=PIPELINE['threshold'],
               ahc_threshold_active=SPEAKER_MODE == 'automatic',
               vad_threshold=PIPELINE['vad_threshold'], window_sec=PIPELINE['window_sec'],
               step_sec=PIPELINE['step_sec'], collar=COLLAR, skip_overlap=SKIP_OVERLAP,
               files=total['n_files'], DER=total['DER'], Miss=total['miss'],
               FalseAlarm=total['false_alarm'], SpeakerError=total['speaker_error'],
               SpeakerCountAccuracy=total['speaker_count_accuracy'],
               SpeakerCountMAE=total['speaker_count_mae'])
with (RESULTS / 'corpus_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary))
    writer.writeheader()
    writer.writerow(summary)
if total['n_files'] != len(dataset):
    raise RuntimeError('Not every recording was scored')
manifest.update(status='scored', files_evaluated=total['n_files'])
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')


In [ ]:
from experiments.viyt_diagnostics import diagnostic_rows, comparison_rows, save_table, REVIEW_IDS

rows = diagnostic_rows(records, references, hypotheses, results, traces)
save_table(rows, RESULTS / 'diagnostics.csv')
(RESULTS / 'diagnostics.json').write_text(json.dumps(rows, indent=2), encoding='utf-8')
review = [row for row in rows if row['audio_id'] in REVIEW_IDS]
save_table(review, RESULTS / 'manual_review.csv')
print('For manual listening (included in corpus):')
for row in review:
    print(row)
if baseline_metrics is not None:
    comparison = comparison_rows(baseline_metrics['__total__'], total)
    save_table(comparison, RESULTS / 'automatic_vs_oracle.csv')
    (RESULTS / 'automatic_vs_oracle.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
    print('Delta = oracle - automatic; DER/component deltas are percentage points.')
    for row in comparison:
        print(row)
    print('Files losing speakers in postprocessing:', sum(r['speakers_lost_in_postprocess'] > 0 for r in rows))
manifest.update(status='complete')
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')


In [ ]:
from IPython.display import display, FileLink

# ZIP содержит результаты/RTTM и точную копию использованного кода; без моделей и WAV.
import zipfile
archive = RUN / f'viyt_model0034_{SPEAKER_MODE}_results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as package:
    for folder in (RESULTS, CODE):
        for path in sorted(folder.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                package.write(path, path.relative_to(RUN))
os.chdir(RUN)
display(FileLink(archive.name))
print('Results:', RESULTS)
print('Сохраните Output notebook или скачайте ZIP. Модели не изменялись.')


## Что прислать после запуска

ZIP с `automatic_vs_oracle.csv`, `corpus_summary.csv`, `diagnostics.csv` и manifest.
Дельта = oracle − automatic (для DER/components в процентных пунктах); baseline читается из реальных результатов.
Три записи в `manual_review.csv` остаются в общей оценке. Короткая reference speech duration сама по себе
не доказывает неполноту разметки; требуется прослушивание.

Oracle — диагностическое условие с дополнительной информацией, а не качество полностью автоматической системы.
Если confusion упадёт, автоматическое число кластеров существенно влияет на ошибку.
Остаточная ошибка не доказывает плохое обучение: нужно учитывать AHC, смешанные окна, постобработку, overlap и reference.
Результаты не оптимизируются по test. Обучение, threshold sweeps и ручная коррекция reference здесь не выполняются.
